# DUCK+: Temporally-Aware Rumour Detection
**CS 6320 — Kishan Rakesh & Annie Johnson Porattoor**

**Before running:** Runtime → Change runtime type → **A100 GPU** + **High-RAM**

Run cells top-to-bottom. Cells 1–4 are one-time setup. Cell 5 onward are the experiments.

## Cell 1 — Anti-idle (run this first, keep this tab open)
Injects a JavaScript keepalive into the browser page so Colab never sees the session as idle. This must be run before anything else.

In [ ]:
# Prevent Colab from disconnecting due to inactivity.
# Clicks the 'Connect' button every 60 seconds via JavaScript.
from IPython.display import Javascript, display

display(Javascript('''
var keepAliveTimer = setInterval(function() {
    console.log('keepAlive ping: ' + new Date().toLocaleTimeString());
    var btn = document.querySelector('colab-connect-button');
    if (btn) {
        var inner = btn.shadowRoot ? btn.shadowRoot.querySelector('button')
                                   : btn.querySelector('button');
        if (inner) inner.click();
    }
}, 60000);
console.log('Anti-idle keepalive started.');
'''))
print('Anti-idle keepalive active. Keep this browser tab open.')


## Cell 2 — Clone repo and set working directory


In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/AnnieJP/duck-rumor-detection'
BRANCH   = 'kishan/duck-plus'
CODE_DIR = '/content/duck-rumor-detection'

if os.path.isdir(os.path.join(CODE_DIR, '.git')):
    subprocess.run(['git', '-C', CODE_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, CODE_DIR], check=True)

os.chdir(CODE_DIR)
print('Working dir:', os.getcwd())
print('Branch:', BRANCH)
print('Data files:', sum(len(fs) for _, _, fs in os.walk('data')))


## Cell 3 — Install dependencies
Skip after first run if packages are already cached in the session.

In [ ]:
import subprocess, torch

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-2000:])
        raise RuntimeError(f'Command failed: {cmd}')
    return result.stdout

# Colab ships PyTorch — detect version and pick matching PyG wheels.
torch_ver = torch.__version__.split('+')[0]   # e.g. '2.5.1'
cuda_tag  = 'cu' + torch.version.cuda.replace('.', '')[:3]  # e.g. 'cu121'
print(f'Colab torch={torch_ver}  cuda_tag={cuda_tag}')

pyg_url = f'https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html'
print(f'PyG wheel index: {pyg_url}')

print('Installing PyG sparse deps...')
run(f'pip install -q torch-scatter torch-sparse -f {pyg_url}')

print('Installing remaining packages...')
run('pip install -q torch-geometric transformers scikit-learn tqdm numpy pandas networkx scipy')

print('All packages installed.')
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}  '
      f'device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')


## Cell 4 — Preprocess raw data → .npz files
**Run once.** Creates `data/twitter15_npz/`, `data/twitter16_npz/`, 5-fold splits, and chronological splits. Safe to re-run (idempotent).

In [ ]:
import os

already_done = (
    os.path.isdir('data/twitter15_npz') and
    os.path.isdir('data/twitter16_npz') and
    len(os.listdir('data/twitter15_npz')) > 100
)

if already_done:
    print('Preprocessing already complete — skipping.')
    print(f'  twitter15: {len(os.listdir("data/twitter15_npz"))} stories')
    print(f'  twitter16: {len(os.listdir("data/twitter16_npz"))} stories')
else:
    print('Running preprocessing...')
    !python preprocess.py --dataset all
    print('Preprocessing complete.')

## Cell 5 — GPU check

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU found. Go to Runtime → Change runtime type → GPU.')

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'torch version: {torch.__version__}')

## Cell 6 — Configure experiment
Edit the variables in this cell to choose what to run.

In [ ]:
# ── What to run ─────────────────────────────────────────────────────────────
# Set to a subset if you want to test one variant first.
VARIANTS = ['baseline', 'temp', 'gated', 'full']   # Table 1 configs
DATASETS = ['twitter15', 'twitter16']
SPLITS   = ['random', 'chrono']
N_FOLDS  = 5    # 5-fold CV for random; 5 seeds for chrono

# ── Hyperparameters ──────────────────────────────────────────────────────────
HID_FEATS    = 64
CT_OUT       = 64
UT_OUT       = 64
LR_BERT      = 2e-5
LR_OTHER     = 1e-3
WEIGHT_DECAY = 5e-5
N_EPOCHS     = 50
BATCH_SIZE   = 8
PATIENCE     = 10
NUM_WORKERS  = 2

RESULT_CSV = 'results/results.csv'
CKPT_DIR   = 'checkpoints'

# ── Build full job list ───────────────────────────────────────────────────────
jobs = [
    (variant, dataset, split, i)
    for variant in VARIANTS
    for dataset in DATASETS
    for split   in SPLITS
    for i       in range(N_FOLDS)
]
print(f'Total jobs: {len(jobs)}')
print('First 4:', jobs[:4])

## Cell 6b — Stage preset (recommended)
Choose one stage to progressively scale from smoke checks to the full experiment. Each stage writes to its own result/checkpoint path to avoid resume collisions.

In [ ]:
# ── Stage preset ladder: smoke -> mini -> realistic -> partial -> full ───────
STAGE = 'smoke'  # one of: smoke, mini, realistic, partial, full

PRESETS = {
    'smoke': {
        'variants': ['full'],
        'datasets': ['twitter15'],
        'splits': ['random'],
        'n_folds': 1,
        'n_epochs': 1,
        'batch_size': 4,
        'smoke_n': 16,
        'result_csv': 'results/smoke_results.csv',
        'ckpt_dir': 'checkpoints/smoke',
    },
    'mini': {
        'variants': ['full'],
        'datasets': ['twitter15'],
        'splits': ['random'],
        'n_folds': 1,
        'n_epochs': 3,
        'batch_size': 4,
        'smoke_n': 64,
        'result_csv': 'results/mini_results.csv',
        'ckpt_dir': 'checkpoints/mini',
    },
    'realistic': {
        'variants': ['full'],
        'datasets': ['twitter15'],
        'splits': ['random', 'chrono'],
        'n_folds': 5,
        'n_epochs': 50,
        'batch_size': 8,
        'smoke_n': None,
        'result_csv': 'results/realistic_results.csv',
        'ckpt_dir': 'checkpoints/realistic',
    },
    'partial': {
        'variants': ['baseline', 'full'],
        'datasets': ['twitter15', 'twitter16'],
        'splits': ['random', 'chrono'],
        'n_folds': 1,
        'n_epochs': 50,
        'batch_size': 8,
        'smoke_n': None,
        'result_csv': 'results/partial_results.csv',
        'ckpt_dir': 'checkpoints/partial',
    },
    'full': {
        'variants': ['baseline', 'temp', 'gated', 'full'],
        'datasets': ['twitter15', 'twitter16'],
        'splits': ['random', 'chrono'],
        'n_folds': 5,
        'n_epochs': 50,
        'batch_size': 8,
        'smoke_n': None,
        'result_csv': 'results/results.csv',
        'ckpt_dir': 'checkpoints',
    },
}

if STAGE not in PRESETS:
    raise ValueError(f'Unknown STAGE={STAGE}. Choose from: {list(PRESETS)}')

cfg = PRESETS[STAGE]

# Override run settings from preset
VARIANTS   = cfg['variants']
DATASETS   = cfg['datasets']
SPLITS     = cfg['splits']
N_FOLDS    = cfg['n_folds']
N_EPOCHS   = cfg['n_epochs']
BATCH_SIZE = cfg['batch_size']
SMOKE_N    = cfg['smoke_n']
RESULT_CSV = cfg['result_csv']
CKPT_DIR   = cfg['ckpt_dir']

# Rebuild job list
jobs = [
    (variant, dataset, split, i)
    for variant in VARIANTS
    for dataset in DATASETS
    for split   in SPLITS
    for i       in range(N_FOLDS)
]

print(f'Stage: {STAGE}')
print(f'Total jobs: {len(jobs)}')
print(f'Variants: {VARIANTS}')
print(f'Datasets: {DATASETS}')
print(f'Splits: {SPLITS}')
print(f'Epochs/job: {N_EPOCHS}, batch size: {BATCH_SIZE}, smoke_n: {SMOKE_N}')
print(f'Results -> {RESULT_CSV}')
print(f'Checkpoints -> {CKPT_DIR}')

## Cell 7 — Run all experiments
Runs every job sequentially. Results are appended to `results/results.csv` after each run so progress is saved even if the session disconnects.

In [ ]:
import os, sys, time, csv, pickle, random
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader

sys.path.insert(0, os.path.join(os.getcwd(), 'model'))
from dataset import DuckPlusDataset
from duck_plus import DuckPlus
from train_duck_plus import (
    set_seed, EarlyStopping, evaluate, append_result, RESULT_COLS
)

device = torch.device('cuda:0')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs('results', exist_ok=True)

# Skip already-completed runs (resume support)
completed = set()
if os.path.exists(RESULT_CSV):
    with open(RESULT_CSV) as f:
        for row in csv.DictReader(f):
            completed.add((row['variant'], row['dataset'],
                           row['split'], int(row['run'])))
    print(f'Found {len(completed)} already-completed runs — skipping them.')

total  = len(jobs)
done   = 0
t_start = time.time()

for variant, dataset, split, fold in jobs:
    key = (variant, dataset, split, fold)
    if key in completed:
        print(f'  SKIP (already done): {key}')
        done += 1
        continue

    print(f'\n[{done+1}/{total}] variant={variant} dataset={dataset} '
          f'split={split} fold/run={fold}')

    seed = 42 + fold
    set_seed(seed)

    # Load split IDs
    npz_dir = f'data/{dataset}_npz'
    if split == 'random':
        fold_dir = f'data/{dataset}_5fold/fold{fold}'
        with open(f'{fold_dir}/_x_train.pkl', 'rb') as f_: train_ids = pickle.load(f_)
        with open(f'{fold_dir}/_x_test.pkl',  'rb') as f_: val_ids   = pickle.load(f_)
        test_ids = val_ids
    else:
        chrono_dir = f'data/{dataset}_chrono'
        with open(f'{chrono_dir}/train.pkl', 'rb') as f_: train_ids = pickle.load(f_)
        with open(f'{chrono_dir}/val.pkl',   'rb') as f_: val_ids   = pickle.load(f_)
        with open(f'{chrono_dir}/test.pkl',  'rb') as f_: test_ids  = pickle.load(f_)

    if SMOKE_N is not None:
        train_ids = train_ids[:SMOKE_N]
        val_ids   = val_ids[:SMOKE_N]
        test_ids  = test_ids[:SMOKE_N]

    train_loader = DataLoader(DuckPlusDataset(train_ids, npz_dir),
                              batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS)
    val_loader   = DataLoader(DuckPlusDataset(val_ids,   npz_dir),
                              batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS)
    test_loader  = DataLoader(DuckPlusDataset(test_ids,  npz_dir),
                              batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS)

    model = DuckPlus(
        variant=variant, hid_feats=HID_FEATS,
        ct_out=CT_OUT, ut_out=UT_OUT, num_classes=4,
    ).to(device)

    # Separate LR for BERT vs rest
    bert_ids = set()
    for branch in ['comment_tree', 'comment_chain', 'user_tree']:
        m = getattr(model, branch, None)
        if m and hasattr(m, 'bert'):
            bert_ids.update(id(p) for p in m.bert.parameters())
    optimizer = torch.optim.Adam([
        {'params': [p for p in model.parameters() if id(p) in bert_ids],
         'lr': LR_BERT},
        {'params': [p for p in model.parameters() if id(p) not in bert_ids],
         'lr': LR_OTHER},
    ], weight_decay=WEIGHT_DECAY)

    ckpt_path = f'{CKPT_DIR}/{variant}_{dataset}_{split}_r{fold}.pt'
    stopper   = EarlyStopping(patience=PATIENCE, ckpt_path=ckpt_path)

    for epoch in range(N_EPOCHS):
        model.train()
        tr_loss, tr_correct, tr_total = [], 0, 0
        for batch in train_loader:
            batch = batch.to(device)
            logits, _ = model(batch)
            labels = batch.y.view(-1)
            loss = F.nll_loss(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tr_loss.append(loss.item())
            tr_correct += logits.argmax(-1).eq(labels).sum().item()
            tr_total   += labels.size(0)

        val_loss, val_acc, val_f1, val_f1_cls = evaluate(model, val_loader, device)
        tr_acc = tr_correct / max(tr_total, 1)
        print(f'  Ep {epoch:03d} '
              f'tr_loss={np.mean(tr_loss):.3f} tr_acc={tr_acc:.3f} '
              f'val_loss={val_loss:.3f} val_f1={val_f1:.3f}')

        stopper(val_f1, val_acc, val_f1_cls, model)
        if stopper.early_stop:
            print(f'  Early stop at epoch {epoch}.')
            break
        torch.cuda.empty_cache()

    # Test
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    _, test_acc, test_f1, test_f1_cls = evaluate(model, test_loader, device)

    result = {
        'variant': variant, 'dataset': dataset, 'split': split, 'run': fold,
        'test_acc': round(test_acc, 4), 'test_macro_f1': round(test_f1, 4),
        'f1_NR': round(test_f1_cls[0], 4), 'f1_FR': round(test_f1_cls[1], 4),
        'f1_TR': round(test_f1_cls[2], 4), 'f1_UR': round(test_f1_cls[3], 4),
        'val_f1': round(stopper.best_f1, 4), 'val_acc': round(stopper.best_acc, 4),
    }
    append_result(result, RESULT_CSV)
    completed.add(key)
    done += 1

    elapsed = (time.time() - t_start) / 60
    remaining = (total - done)
    avg_per_job = elapsed / done if done else 0
    print(f'  TEST acc={test_acc:.4f} macro-F1={test_f1:.4f} '
          f'NR={test_f1_cls[0]:.4f} FR={test_f1_cls[1]:.4f} '
          f'TR={test_f1_cls[2]:.4f} UR={test_f1_cls[3]:.4f}')
    print(f'  Progress: {done}/{total} done | '
          f'~{avg_per_job*remaining:.0f} min remaining')

    del model, optimizer
    torch.cuda.empty_cache()

print(f'\nAll done! Results saved to {RESULT_CSV}')

## Cell 8 — Aggregate and display results table

In [ ]:
import pandas as pd

df = pd.read_csv(RESULT_CSV)
print(f'Total rows: {len(df)}\n')

summary = (
    df.groupby(['variant', 'dataset', 'split'])
      [['test_macro_f1', 'test_acc', 'f1_NR', 'f1_FR', 'f1_TR', 'f1_UR']]
      .agg(['mean', 'std'])
      .round(4)
)
print(summary.to_string())

## Cell 9 — Plot: macro-F1 by variant and split type

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv(RESULT_CSV)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
variant_order = ['baseline', 'temp', 'gated', 'full']
colors = {'random': '#4C72B0', 'chrono': '#DD8452'}

for ax, dataset in zip(axes, ['twitter15', 'twitter16']):
    sub = df[df['dataset'] == dataset]
    means = sub.groupby(['variant', 'split'])['test_macro_f1'].mean().unstack('split')
    stds  = sub.groupby(['variant', 'split'])['test_macro_f1'].std().unstack('split')
    means = means.reindex(variant_order)
    stds  = stds.reindex(variant_order)

    x = np.arange(len(variant_order))
    w = 0.35
    for i, split in enumerate(['random', 'chrono']):
        if split in means.columns:
            ax.bar(x + i*w, means[split], w,
                   yerr=stds[split], capsize=4,
                   label=split, color=colors[split], alpha=0.85)

    ax.set_title(dataset.capitalize(), fontsize=13)
    ax.set_xticks(x + w/2)
    ax.set_xticklabels(variant_order)
    ax.set_ylabel('Macro-F1')
    ax.set_ylim(0, 1.0)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('DUCK+ Factorial Ablation — Macro-F1 by Variant and Split', fontsize=14)
plt.tight_layout()
plt.savefig('results/macro_f1_plot.png', dpi=150)
plt.show()
print('Plot saved to results/macro_f1_plot.png')

## Cell 10 — Inspect learned gate values (full variant only)

In [ ]:
import sys, os, pickle, torch
import numpy as np
sys.path.insert(0, os.path.join(os.getcwd(), 'model'))
from dataset import DuckPlusDataset
from duck_plus import DuckPlus
from torch_geometric.loader import DataLoader

# Reads dims from Cell 6 — change these only if you changed Cell 6.
GATE_DATASET = 'twitter15'
GATE_FOLD    = 0
ckpt_path    = f'checkpoints/full_{GATE_DATASET}_random_r{GATE_FOLD}.pt'

if not os.path.exists(ckpt_path):
    print(f'Checkpoint not found: {ckpt_path}. Run Cell 7 first.')
else:
    with open(f'data/{GATE_DATASET}_5fold/fold{GATE_FOLD}/_x_test.pkl', 'rb') as f:
        test_ids = pickle.load(f)

    loader = DataLoader(
        DuckPlusDataset(test_ids, f'data/{GATE_DATASET}_npz'),
        batch_size=8, shuffle=False, num_workers=2
    )
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

    # Use same dims as Cell 6 so checkpoint loads without shape mismatch.
    model = DuckPlus(
        variant='full',
        hid_feats=HID_FEATS,   # from Cell 6
        ct_out=CT_OUT,         # from Cell 6
        ut_out=UT_OUT,         # from Cell 6
    ).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    all_gates = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            _, gates = model(batch)
            if gates is not None:
                all_gates.append(gates.cpu().numpy())

    all_gates = np.concatenate(all_gates, axis=0)  # (N_stories, 3)
    branch_names = ['comment_tree (g1)', 'comment_chain (g2)', 'user_tree (g3)']
    print(f'Gate statistics across {len(all_gates)} test stories:')
    print(f'  {"Branch":<25} {"Mean":>8} {"Std":>8} {"Min":>8} {"Max":>8}')
    print('  ' + '-'*57)
    for i, name in enumerate(branch_names):
        g = all_gates[:, i]
        print(f'  {name:<25} {g.mean():8.4f} {g.std():8.4f} '
              f'{g.min():8.4f} {g.max():8.4f}')
